In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
instructions = parser.get_format_instructions()
print(instructions)

Return a JSON object.


In [3]:
ai_response = '{"이름": "김철수", "나이": 30}'
parsed_response = parser.parse(ai_response)
print(parsed_response)
print(type(parsed_response))

{'이름': '김철수', '나이': 30}
<class 'dict'>


In [7]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, model_validator

model = ChatOpenAI(model="gpt-4o", temperature=0)

class FinancialAdvice(BaseModel):
    setup: str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice: str = Field(description="질문을 해결하기 위한 금융 답변")

    @model_validator(mode="before")
    @classmethod
    def question_ends_with_question_mars(cls, values: dict) -> dict:
        setup = values.get("setup", "")
        if not setup.endswith("?"):
            raise ValueError("잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다.")
        return values

parser = PydanticOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate.from_template(
    template="""
    다음 금융 관련 질문에 답변해 주세요.
    {format_instructions}
    질문: {query}

    """,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | model | parser

# 체인 실행 및 결과 출력
try:
    result = chain.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있는 질문하여라."})
    print(result)
except Exception as e:
    print(f"오류 발생: {e}")

setup='부동산 투자에 대한 조언을 받고 싶습니다. 현재 부동산 시장의 전망과 투자 시 고려해야 할 요소는 무엇인가요?' advice='부동산 투자를 고려할 때는 시장의 현재 상태와 미래 전망을 분석하는 것이 중요합니다. 지역 경제 성장, 인구 증가, 인프라 개발 계획 등을 조사하여 해당 지역의 부동산 가치 상승 가능성을 평가하세요. 또한, 금리 변동, 세금 정책, 대출 조건 등 금융 환경도 고려해야 합니다. 전문가와 상담하여 포트폴리오 다각화와 리스크 관리 전략을 수립하는 것도 좋은 방법입니다.'


In [8]:
print(prompt.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있는 질문하여라."}).to_string())


    다음 금융 관련 질문에 답변해 주세요.
    The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"setup": {"description": "금융 조언 상황을 설정하기 위한 질문", "title": "Setup", "type": "string"}, "advice": {"description": "질문을 해결하기 위한 금융 답변", "title": "Advice", "type": "string"}}, "required": ["setup", "advice"]}
```
    질문: 부동산에 관련하여 금융 조언을 받을 수 있는 질문하여라.

    


In [9]:
print(type(result))

<class '__main__.FinancialAdvice'>


In [10]:
print(model.invoke(prompt.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있는 질문하여라."})).content)

```json
{
  "setup": "부동산 투자에 대한 조언을 받고 싶습니다. 현재 시장 상황에서 부동산에 투자하는 것이 좋은 선택인지, 그리고 어떤 유형의 부동산에 투자하는 것이 가장 유리한지 알고 싶습니다.",
  "advice": "현재 부동산 시장은 지역에 따라 다르게 움직이고 있습니다. 투자하기 전에 지역 시장 동향을 철저히 조사하는 것이 중요합니다. 주거용 부동산은 안정적인 수익을 제공할 수 있지만, 상업용 부동산은 더 높은 수익률을 제공할 수 있습니다. 또한, 부동산 투자 신탁(REITs)과 같은 간접 투자 방법도 고려해 볼 수 있습니다. 전문가와 상담하여 자신의 재정 상황과 목표에 맞는 최적의 투자 전략을 수립하는 것이 좋습니다."
}
```


In [11]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = PromptTemplate.from_template(
    "다음 질문에 대한 답변이 포함된 JSON 객체를 반환하십시오: {question}"
)

json_parser = JsonOutputParser()
json_chain = json_prompt | model | json_parser

list(json_chain.stream({"question": "비트코인에 대한 짧은 한문장 설명."}))

[{},
 {'description': ''},
 {'description': '비'},
 {'description': '비트'},
 {'description': '비트코'},
 {'description': '비트코인은'},
 {'description': '비트코인은 분'},
 {'description': '비트코인은 분산'},
 {'description': '비트코인은 분산형'},
 {'description': '비트코인은 분산형 디'},
 {'description': '비트코인은 분산형 디지털'},
 {'description': '비트코인은 분산형 디지털 화'},
 {'description': '비트코인은 분산형 디지털 화폐'},
 {'description': '비트코인은 분산형 디지털 화폐로'},
 {'description': '비트코인은 분산형 디지털 화폐로,'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간의'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간의 거래'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간의 거래를'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간의 거래를 가능'},
 {'description': '비트코인은 분산형 디지털 화폐로, 중앙은행 없이 개인 간의 거래를 가능하게'},
 {'description': '비트코인은 분산형 디

In [14]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

model = ChatOpenAI(temperature=0)

class FinancialAdvice(BaseModel):
    setup: str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice: str = Field(description="질문을 해결하기 위한 금융 답변")

parser = JsonOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate(
    template="다음 금융 관련 질문에 답변해 주세요.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | model | parser
chain.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있는 질문하여라."})

{'setup': '부동산 투자에 대한 금융 조언을 받고 싶습니다.',
 'advice': '부동산 시장의 현재 상황을 파악하고 투자 전략을 수립하는 것이 중요합니다. 또한 임대 수익과 가치 상승을 고려하여 장기적인 투자 계획을 세우는 것이 좋습니다.'}